# Con cámara fija en un cuerpo

In [ ]:
from IPython.display import HTML
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from scipy.integrate import solve_ivp


mu = 5 # = G * m1
x0, y0, z0 = 1, 0, 0
vx0, vy0, vz0 = 0, 1, 0

def sist_eq(t, X):
    x, vx, y, vy, z, vz = X
    r3 = (x**2 + y**2 + z**2)**1.5
    dxdt = vx
    dydt = vy
    dzdt = vz
    dvxdt = -mu * x / r3
    dvydt = -mu * y / r3
    dvzdt = -mu * z / r3
    return [dxdt, dvxdt, dydt, dvydt, dzdt, dvzdt]

X0 = [x0, vx0, y0, vy0, z0, vz0]
t_total = (0, 10)
t_eval = np.linspace(*t_total, 10000) #unpack la tupla
sol = solve_ivp(sist_eq, t_total, X0, t_eval=t_eval, rtol=1e-9, atol=1e-12)


step = 1 # bajamos frames pa que no pete
x_t = sol.y[0][::step]
y_t = sol.y[2][::step]
z_t = sol.y[4][::step]


def animate_orbit():
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    
    lim = max(np.max(np.abs(x_t)), np.max(np.abs(y_t)), np.max(np.abs(z_t)))# maximos par ajsutar limites plot
    ax.set_xlim([-lim*1.1, lim*1.1])
    ax.set_ylim([-lim*1.1, lim*1.1])
    ax.set_zlim([-lim*1.1, lim*1.1])
    ax.scatter(0, 0, 0, color='red', s=50)

    line, = ax.plot([], [], [], color='blue', lw=1)
    point, = ax.plot([], [], [], 'o', color='blue')

    def update(num):
        line.set_data(x_t[:num], y_t[:num])
        line.set_3d_properties(z_t[:num])
        point.set_data(x_t[num-1:num], y_t[num-1:num])
        point.set_3d_properties(z_t[num-1:num])
        return line, point

    ani = FuncAnimation(fig, update, frames=len(x_t), interval=speed_anim, blit=False)
    return ani

speed_anim = 500
ani = animate_orbit()
#HTML(ani.to_jshtml())
ani.save("outputs/orbita3D.gif", writer="pillow", fps=50)


# Modificación sin camara fija en un cuerpo

In [9]:
# -*- coding: utf-8 -*-
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# ---------- Parámetros físicos ----------
G = 1.0           # constante gravitatoria (unidad arbitraria)
m1 = 5.0          # masa 1 (puedes cambiar)
m2 = 3.0          # masa 2 (puedes cambiar)
M = m1 + m2       # masa total

# ---------- Condiciones iniciales para el vector relativo r = r2 - r1 ----------
# Ejemplo: separación inicial en x=1, velocidad relativa en y=1
rx0, ry0, rz0 = 1.0, 0.0, 0.0   # r(0)
vx0, vy0, vz0 = 0.0, 1.0, 0.0   # r'(0)

# Estado inicial: [rx, ry, rz, vx, vy, vz]
X0 = [rx0, ry0, rz0, vx0, vy0, vz0]

# ---------- Ecuación para el vector relativo ----------
def rel_equations(t, X):
    rx, ry, rz, vx, vy, vz = X
    rvec = np.array([rx, ry, rz])
    rnorm = np.linalg.norm(rvec)
    # evitar división por cero
    if rnorm == 0:
        a = np.array([0.0, 0.0, 0.0])
    else:
        a = - G * M * rvec / (rnorm**3)   # r'' = -G*(m1+m2) * r / |r|^3
    return [vx, vy, vz, a[0], a[1], a[2]]

# ---------- Integración ----------
t_span = (0.0, 10.0)             # tiempo total
n_frames = 200                    # número de puntos a guardar (ajustable)
t_eval = np.linspace(t_span[0], t_span[1], n_frames)

sol = solve_ivp(rel_equations, t_span, X0, t_eval=t_eval, rtol=1e-9, atol=1e-12)

# extraemos r(t)
rx_t = sol.y[0]
ry_t = sol.y[1]
rz_t = sol.y[2]

# ---------- Posiciones de los dos cuerpos en el sistema centrado en el punto medio ----------
# punto medio geométrico = (r1 + r2)/2, al desplazar por él obtenemos posiciones +/- r/2
x1_mid = -rx_t / 2.0
y1_mid = -ry_t / 2.0
z1_mid = -rz_t / 2.0

x2_mid = +rx_t / 2.0
y2_mid = +ry_t / 2.0
z2_mid = +rz_t / 2.0

# Si además quieres la posición de 2 respecto a 1 (vector relativo), es simplemente r(t):
# r_rel = (rx_t, ry_t, rz_t). Y la de 1 respecto a 2 = -r(t).

# ---------- Función de animación ----------
def make_animation(save_path="orbits_midpoint.gif", fps=30):
    fig = plt.figure(figsize=(6,6))
    ax = fig.add_subplot(111, projection='3d')

    # límites fijos (cámara fija mirando al origen = punto medio)
    lim = 1.1 * max(np.max(np.abs(rx_t))/2.0, 0.5)  # /2 porque usamos +/- r/2
    ax.set_xlim([-lim, lim])
    ax.set_ylim([-lim, lim])
    ax.set_zlim([-lim, lim])

    # mantenemos la vista fija (cámara fija)
    ax.view_init(elev=20, azim=30)  # elevación y azimut fijos

    # dibujamos el punto medio en el origen
    ax.scatter(0, 0, 0, color='k', s=30, label='punto medio')

    line1, = ax.plot([], [], [], 'o-', lw=1, label='cuerpo 1', markersize=6)
    line2, = ax.plot([], [], [], 'o-', lw=1, label='cuerpo 2', markersize=6)
    # línea que une los cuerpos
    connector, = ax.plot([], [], [], '--', lw=0.8, alpha=0.7)

    ax.legend(loc='upper right')

    def update(i):
        # trayectorias hasta i (opcional: puedes mostrar solo el punto)
        # aquí muestro solo los puntos y la línea que los une
        x1 = x1_mid[i]
        y1 = y1_mid[i]
        z1 = z1_mid[i]
        x2 = x2_mid[i]
        y2 = y2_mid[i]
        z2 = z2_mid[i]

        line1.set_data([x1], [y1])
        line1.set_3d_properties([z1])
        line2.set_data([x2], [y2])
        line2.set_3d_properties([z2])

        connector.set_data([x1, x2], [y1, y2])
        connector.set_3d_properties([z1, z2])

        return line1, line2, connector

    ani = FuncAnimation(fig, update, frames=len(rx_t), interval=1000/fps, blit=False)
    ani.save(save_path, writer='pillow', fps=fps)
    plt.close(fig)
    print(f"Guardado GIF en: {save_path}")

# Para ejecutar y guardar:
make_animation("outputs/orbit_midpoint.gif", fps=5)

# ---------- Ejemplo: posiciones relativas (arrays) ----------
# r(t) = (rx_t, ry_t, rz_t)  -> 2 respecto a 1
# r_2_wrt_1 = np.vstack([rx_t, ry_t, rz_t]).T
# r_1_wrt_2 = -r_2_wrt_1


Guardado GIF en: outputs/orbit_midpoint.gif
